# Part 1 — Does DoRA's weight-decomposition claim hold in a VLM's vision encoder?

EN4554 Assignment 1. SmolVLM-256M-Instruct, free Colab T4.

**Runtime → Change runtime type → T4 GPU** before running anything.

Order of operations:
1. Setup + sanity checks (~5 min) — never skip these, they catch silent mis-targeting
2. Smoke run (~5 min) — proves the pipeline end to end on 64 examples
3. Tier 1 (~2.5 h) — the vision-tower replication of DoRA Figure 2; independently submittable
4. Tiers 2–4 — one hypothesis each; run in order, stop when the quota stops you
5. Analysis + figures

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# Colab ships a working torch for this runtime -- do NOT reinstall it.
# Exact pins: Colab and Kaggle must draw the identical TextVQA subset (see requirements.txt).
!pip -q install "transformers==5.23.0" "peft==0.21.3" "datasets==5.2.0" "accelerate==1.15.0" scipy num2words
import torch, transformers, peft, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| datasets", datasets.__version__)
assert transformers.__version__ == "5.23.0" and datasets.__version__ == "5.2.0", \
    "Old versions still loaded: Runtime -> Restart session, then run from this cell again"

In [ ]:
import os, sys
REPO = "/content/LoRA-finetuning"

# The GitHub repo is private: add a token under Colab's key icon (Secrets) named
# GITHUB_TOKEN, with "Notebook access" on. A public repo needs no token.
try:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
except Exception:
    _tok = None
URL = (f"https://{_tok}@github.com/minuraashen/LoRA-finetuning.git" if _tok
       else "https://github.com/minuraashen/LoRA-finetuning.git")
if not os.path.exists(REPO):
    !git clone -q {URL} {REPO}
else:
    !git -C {REPO} pull -q --ff-only          # pick up fixes pushed since the last session
os.chdir(REPO); sys.path.insert(0, os.path.join(REPO, "src"))

# Durability: results land in Drive so a disconnect costs one run, never the batch.
from google.colab import drive; drive.mount('/content/drive')
os.environ["RESULTS_DIR"] = "/content/drive/MyDrive/en4554_dora/results"
os.environ["FIG_DIR"]     = "/content/drive/MyDrive/en4554_dora/figures"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # keeps the cell output readable
os.makedirs(os.environ["RESULTS_DIR"], exist_ok=True)
os.makedirs(os.environ["FIG_DIR"], exist_ok=True)

# Which slice of the 26 runs this machine owns. The parallel plan is 5 shards:
#   Kaggle session 1 -> 0/5 and 1/5 (one per T4)    Colab -> 2/5 (this notebook)
#   Kaggle session 2 -> 3/5 and 4/5
# Running everything on Colab alone? Set SHARD = "".
SHARD = "2/5"
SHARD_ARG = f"--shard {SHARD}" if SHARD else ""

print(os.environ["RESULTS_DIR"])
print("completed runs so far:", sorted(f for f in os.listdir(os.environ["RESULTS_DIR"]) if f.endswith(".json")))
!python run_grid.py --tiers 1 2 3 4 {SHARD_ARG} --list

## 1. Sanity checks

These verify, in order: the dM/dD math; that the magnitude axis is per **output**
feature (what PEFT actually trains); that the regex matches all 72 vision Linears
including `out_proj`/`fc1`/`fc2`; that magnitude-only has exactly 82,944 trainable
parameters; that a DoRA-wrapped model is **bit-identical** to the base model at init;
that every image costs exactly 64 tokens; and that the loss covers the answer only.

In [ ]:
!python tests/test_sanity.py

## 2. Smoke run

64 train / 32 eval examples. The point is not the numbers — it is that the loss moves
and the dM/dD scatter is non-degenerate before any real quota is spent.

In [ ]:
!python run_grid.py --tiers 1 --smoke --methods lora --datasets pathvqa --overwrite

In [ ]:
import json, glob, os
# Smoke records go to a separate folder so they can never be mistaken for real runs.
smoke_dir = os.environ["RESULTS_DIR"].rstrip("/") + "_smoke"
rec = json.load(open(sorted(glob.glob(os.path.join(smoke_dir, "lora_V_pathvqa*.json")))[0]))
print("loss:", [round(h["loss"], 4) for h in rec["train"]["loss_history"]])
print("headline:", rec["accuracy"])
print("decomposition rows:", len(rec["decomposition"]))
print("sample:", rec["decomposition"][:2])
print("predictions:", rec["sample_predictions"][:5])
print("minutes for 64 examples:", round(rec["wall_clock_s"] / 60, 1))

## 3. Tier 1 — H1 + H3 (6 runs, ~2.5 h)

Full FT / LoRA / DoRA on the **vision encoder only**, on PathVQA (domain shift, verified
absent from The Cauldron) and TextVQA (in-domain, verified present).

Expect accuracy to barely move here: the language model is frozen, so there is no adapted
task head. That is by design — this tier exists to measure dM/dD under a real gradient.
All accuracy claims come from Tier 3.

In [ ]:
!python run_grid.py --tiers 1 {SHARD_ARG}

## 4. Tier 2 — H2 (vision vs language, same data)

In [ ]:
!python run_grid.py --tiers 2 {SHARD_ARG}

## 5. Tier 3 — accuracy claims + H5 (includes the seed repeats)

In [ ]:
!python run_grid.py --tiers 3 {SHARD_ARG}

## 6. Tier 4 — H4, the bonus

`magonly` = DoRA with lora_A/lora_B frozen, i.e. a learned per-output-channel gain
(82,944 params, 0.03% of the model). `magV_doraL` is DoRA sec. 5.6 / Table 6 translated
to a VLM: magnitude-only in the vision tower, full DoRA in the language model.

In [ ]:
!python run_grid.py --tiers 4 {SHARD_ARG}

## 7. Analysis and figures

In [ ]:
import importlib, analyze
importlib.reload(analyze)
res = analyze.run_all(os.environ["RESULTS_DIR"])
display(res["summary"])
# Headline: query matrices only, output-feature axis -- the paper's Figure 2 setting.
display(res["correlations"].query("axis == 'out' and matrix_set == 'q'")[["run_id","group","r","slope","ft_likeness","headline"]])
print(json.dumps(res["h5"], indent=2))

In [ ]:
from IPython.display import Image, display as d
for f in res["figures"]:
    print(f); d(Image(f))

## 8. What to write up

- **H1** — Fig. 1 top row: do the vision-encoder correlation signs match the paper's
  (FT −0.62 / LoRA +0.83 / DoRA −0.31)? Those reference values are printed on each panel.
- **H2** — Fig. 1 top vs bottom row. If the language row reproduces the paper's signs but
  the vision row does not, that is the finding. If *neither* row reproduces them, suspect
  the pipeline before the hypothesis.
- **H3** — Fig. 2: is PathVQA shifted right (more direction) relative to TextVQA?
- **H4** — Fig. 3: where does the 82,944-parameter point sit on each dataset?
- **H5** — `h5.json`. n is 8–12, so report Spearman with its p-value and call it
  suggestive. A null result here is the most interesting outcome available at this scale.

Also report: the seed, the exact subset sizes and splits, that `do_image_splitting=False`
caps image detail, and that the literature check is "searched and did not find", not
"does not exist".